# Demostración del Agente de Enrutamiento de Inventario (Módulo A)
Este notebook demuestra el funcionamiento del algoritmo A* para la optimización de rutas de recolección de productos dentro del almacén.

In [4]:
# Importar nuestras clases (asumiendo que están en src/search_csp)
# Para esta prueba, copiaremos las clases agent y algorithm aquí mismo 
# si no has configurado los paths de importación en Jupyter, 
# o puedes importarlas si tu entorno está bien configurado:
# from src.search_csp.agent import WarehouseEnvironment, InventoryAgent
# from src.search_csp.algorithm import a_star_search, manhattan_distance
import heapq

# --- COPIA AQUÍ LAS CLASES agent.py y algorithm.py PARA LA PRUEBA RÁPIDA ---
class WarehouseEnvironment:
    def __init__(self, grid, start, goal):
        """
        grid: Matriz 2D donde 0 es pasillo libre y 1 es estantería (obstáculo).
        start: Tupla (x, y) con la posición inicial.
        goal: Tupla (x, y) con la ubicación del producto a recolectar.
        """
        self.grid = grid
        self.start = start
        self.goal = goal
        self.rows = len(grid)
        self.cols = len(grid[0])

    def is_valid_state(self, state):
        x, y = state
        # Verifica límites del grid y que no sea un obstáculo
        if 0 <= x < self.rows and 0 <= y < self.cols:
            return self.grid[x][y] == 0
        return False

    def get_successors(self, state):
        x, y = state
        successors = []
        # Movimientos: Arriba, Abajo, Izquierda, Derecha
        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        
        for dx, dy in moves:
            next_state = (x + dx, y + dy)
            if self.is_valid_state(next_state):
                # El costo de moverse un bloque es siempre 1
                successors.append((next_state, 1)) 
        return successors

class InventoryAgent:
    def __init__(self, environment):
        self.env = environment

    def get_start_state(self):
        return self.env.start

    def is_goal(self, state):
        return state == self.env.goal
import heapq

def manhattan_distance(state, goal):
    """
    Función heurística h(n): Calcula la distancia Manhattan entre dos puntos.
    Es admisible porque nunca sobreestima la distancia real en un grid que solo permite movimientos ortogonales.
    """
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])

def a_star_search(agent):
    """
    Implementación del algoritmo A* usando f(n) = g(n) + h(n)
    """
    start_state = agent.get_start_state()
    goal_state = agent.env.goal
    
    # Priority Queue: almacena tuplas (f_score, state)
    frontier = []
    heapq.heappush(frontier, (0, start_state))
    
    # Diccionarios para rastrear el camino y el costo real g(n)
    came_from = {start_state: None}
    cost_so_far = {start_state: 0}
    
    while frontier:
        current_f, current_state = heapq.heappop(frontier)
        
        if agent.is_goal(current_state):
            # Reconstruir el camino si llegamos a la meta
            path = []
            while current_state is not None:
                path.append(current_state)
                current_state = came_from[current_state]
            path.reverse()
            return path
            
        for next_state, cost in agent.env.get_successors(current_state):
            new_cost = cost_so_far[current_state] + cost
            
            # Si no hemos visitado el estado o encontramos un camino más barato
            if next_state not in cost_so_far or new_cost < cost_so_far[next_state]:
                cost_so_far[next_state] = new_cost
                priority = new_cost + manhattan_distance(next_state, goal_state)
                heapq.heappush(frontier, (priority, next_state))
                came_from[next_state] = current_state
                
    return None # Retorna None si no hay camino posible
# 1. Definir un pequeño caso de prueba (El almacén)
# 0 = Pasillo libre, 1 = Estantería (obstáculo)
test_grid = [
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0]
]

# Definir punto de inicio (ej. Muelle de carga) y la meta (ej. Estante del producto)
start_pos = (0, 0)
goal_pos = (4, 4)

In [5]:
# 2. Instanciar el entorno y el agente
env = WarehouseEnvironment(test_grid, start_pos, goal_pos)
agent = InventoryAgent(env)

# 3. Ejecutar A*
path = a_star_search(agent)

print("--- Resultados de la Búsqueda A* ---")
if path:
    print(f"Ruta encontrada! Longitud: {len(path)} pasos.")
    print(f"Camino: {path}")
else:
    print("No se encontró una ruta posible.")

--- Resultados de la Búsqueda A* ---
Ruta encontrada! Longitud: 9 pasos.
Camino: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4)]


In [6]:
# 4. Visualizar el camino en el grid
def print_grid_with_path(grid, path, start, goal):
    for i in range(len(grid)):
        row_str = ""
        for j in range(len(grid[0])):
            pos = (i, j)
            if pos == start:
                row_str += " S " # Start
            elif pos == goal:
                row_str += " G " # Goal
            elif path and pos in path:
                row_str += " * " # Camino
            elif grid[i][j] == 1:
                row_str += "[X]" # Obstáculo
            else:
                row_str += " . " # Pasillo libre
        print(row_str)

print("\nVisualización del Almacén:")
print("S = Inicio, G = Meta, * = Ruta, [X] = Estantería, . = Pasillo")
print_grid_with_path(test_grid, path, start_pos, goal_pos)


Visualización del Almacén:
S = Inicio, G = Meta, * = Ruta, [X] = Estantería, . = Pasillo
 S  *  *  *  * 
 . [X][X][X] * 
 .  .  . [X] * 
[X][X] . [X] * 
 .  .  .  .  G 
